In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
train_path = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/train"

val_path = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/val"

test_path = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/test"

In [2]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator

from tensorflow.keras.applications import MobileNetV2

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Dense,
    Dropout,
    GlobalAveragePooling2D
)

from tensorflow.keras.callbacks import EarlyStopping

from tensorflow.keras.optimizers import Adam

from sklearn.utils.class_weight import compute_class_weight

from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

2026-05-19 10:52:09.961788: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779187930.360936      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779187930.474431      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779187931.579822      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779187931.579863      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779187931.579865      57 computation_placer.cc:177] computation placer alr

In [3]:

train_datagen = ImageDataGenerator(

    rescale=1./255,

    rotation_range=5,

    zoom_range=0.05,

    width_shift_range=0.05,
    height_shift_range=0.05,

    horizontal_flip=True,

    validation_split=0.2
)

test_datagen = ImageDataGenerator(
    rescale=1./255
)

In [4]:
train_generator = train_datagen.flow_from_directory(

    train_path,

    target_size=(224,224),

    batch_size=32,

    class_mode='binary',

    subset='training'
)

Found 4173 images belonging to 2 classes.


In [5]:
val_generator = train_datagen.flow_from_directory(

    train_path,

    target_size=(224,224),

    batch_size=32,

    class_mode='binary',

    subset='validation'
)

Found 1043 images belonging to 2 classes.


In [6]:
test_generator = test_datagen.flow_from_directory(

    test_path,

    target_size=(224,224),

    batch_size=32,

    class_mode='binary',

    shuffle=False
)

Found 624 images belonging to 2 classes.


In [7]:
print(train_generator.class_indices)

{'NORMAL': 0, 'PNEUMONIA': 1}


In [8]:
class_weights = compute_class_weight(

    class_weight='balanced',

    classes=np.unique(train_generator.classes),

    y=train_generator.classes
)

class_weights = dict(enumerate(class_weights))

print(class_weights)

{0: np.float64(1.9445479962721341), 1: np.float64(0.6730645161290323)}


In [9]:
early_stop = EarlyStopping(

    monitor='val_loss',

    patience=4,

    restore_best_weights=True
)

In [10]:
base_model = MobileNetV2(

    weights='imagenet',

    include_top=False,

    input_shape=(224,224,3)
)

I0000 00:00:1779187999.769012      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779187999.774977      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [11]:
model = Sequential()

model.add(base_model)

model.add(GlobalAveragePooling2D())

model.add(Dense(128, activation='relu'))

model.add(Dropout(0.5))

model.add(Dense(1, activation='sigmoid'))

In [12]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 2,387,969 (9.11 MB)

 Non-trainable params: 34,112 (133.25 KB)

In [13]:
base_model.trainable = False

In [14]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [15]:
model.compile(
    optimizer = Adam(learning_rate = 0.0001),
    loss = 'binary_crossentropy',
    metrics=['accuracy']
)

In [16]:
model.fit(
    train_generator,
    validation_data = val_generator,
    epochs = 10,
    class_weight = class_weights,
    callbacks = [early_stop]
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10


I0000 00:00:1779188116.672670     155 service.cc:152] XLA service 0x7c65600112c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779188116.672715     155 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1779188116.672720     155 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1779188117.896141     155 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-05-19 10:55:27.839808: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 10:55:27.995049: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 10:55:28.132959: E external/local_xl

 37/131 ━━━━━━━━━━━━━━━━━━━━ 1:08 732ms/step - accuracy: 0.6135 - loss: 0.7217

2026-05-19 10:56:05.685396: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 10:56:05.821511: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 10:56:05.958754: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 840ms/step - accuracy: 0.7374 - loss: 0.5260

2026-05-19 10:57:56.591718: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 10:57:56.728687: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


131/131 ━━━━━━━━━━━━━━━━━━━━ 168s 1s/step - accuracy: 0.7381 - loss: 0.5248 - val_accuracy: 0.9147 - val_loss: 0.2166
Epoch 2/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 89s 679ms/step - accuracy: 0.9113 - loss: 0.2159 - val_accuracy: 0.9041 - val_loss: 0.2082
Epoch 3/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 96s 731ms/step - accuracy: 0.9226 - loss: 0.1872 - val_accuracy: 0.9214 - val_loss: 0.1761
Epoch 4/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 89s 680ms/step - accuracy: 0.9393 - loss: 0.1510 - val_accuracy: 0.9108 - val_loss: 0.2108
Epoch 5/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 88s 675ms/step - accuracy: 0.9419 - loss: 0.1385 - val_accuracy: 0.9386 - val_loss: 0.1480
Epoch 6/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 88s 675ms/step - accuracy: 0.9474 - loss: 0.1304 - val_accuracy: 0.9569 - val_loss: 0.1225
Epoch 7/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 89s 677ms/step - accuracy: 0.9450 - loss: 0.1400 - val_accuracy: 0.9492 - val_loss: 0.1296
Epoch 8/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 89s 683ms/step - accuracy: 0.9500 - loss: 0.1244 - val_a

In [17]:
test_generator.reset()

y_prob = model.predict(test_generator)

y_pred = (y_prob > 0.5).astype(int)

y_true = test_generator.classes

19/20 ━━━━━━━━━━━━━━━━━━━━ 0s 393ms/step

2026-05-19 11:12:10.580099: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-19 11:12:10.717472: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


20/20 ━━━━━━━━━━━━━━━━━━━━ 23s 947ms/step


In [18]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.93      0.79      0.85       234
           1       0.88      0.96      0.92       390

    accuracy                           0.90       624
   macro avg       0.90      0.88      0.89       624
weighted avg       0.90      0.90      0.90       624



In [19]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_true, y_pred))

[[185  49]
 [ 15 375]]


In [20]:
base_model.trainable = True

In [21]:
for layer in base_model.layers[:-30]:
    layer.trainable = False

In [22]:
from tensorflow.keras.optimizers import Adam

model.compile(

    optimizer=Adam(learning_rate=1e-5),

    loss='binary_crossentropy',

    metrics=['accuracy']
)

In [23]:
history_fine = model.fit(

    train_generator,

    epochs=5,

    validation_data=val_generator,

    class_weight=class_weights,

    callbacks=[early_stop]
)

Epoch 1/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 119s 789ms/step - accuracy: 0.9128 - loss: 0.2836 - val_accuracy: 0.9549 - val_loss: 0.1106
Epoch 2/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 90s 685ms/step - accuracy: 0.9366 - loss: 0.1454 - val_accuracy: 0.9655 - val_loss: 0.0964
Epoch 3/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 89s 683ms/step - accuracy: 0.9496 - loss: 0.1273 - val_accuracy: 0.9626 - val_loss: 0.0936
Epoch 4/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 90s 687ms/step - accuracy: 0.9516 - loss: 0.1195 - val_accuracy: 0.9722 - val_loss: 0.0784
Epoch 5/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 92s 705ms/step - accuracy: 0.9614 - loss: 0.1002 - val_accuracy: 0.9789 - val_loss: 0.0650


In [24]:
test_generator.reset()

y_prob = model.predict(test_generator)

y_pred = (y_prob > 0.6).astype(int)

y_true = test_generator.classes

20/20 ━━━━━━━━━━━━━━━━━━━━ 13s 434ms/step


In [25]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.93      0.78      0.85       234
           1       0.88      0.97      0.92       390

    accuracy                           0.90       624
   macro avg       0.91      0.87      0.89       624
weighted avg       0.90      0.90      0.90       624



In [26]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_true, y_pred))

[[183  51]
 [ 13 377]]


In [27]:
model.save("pneumonia_mobilenetv2.keras")

In [28]:
import os

os.listdir('/kaggle/working')

['pneumonia_mobilenetv2.keras', '.virtual_documents']

In [29]:
import shutil

shutil.make_archive(
    "pneumonia_model",
    'zip',
    '/kaggle/working'
)

'/kaggle/working/pneumonia_model.zip'